# Análise de dados

In [ ]:
import pandas as pd

In [ ]:
url = "http://127.0.0.1:5000/acessos"

dados = pd.read_json(url)

dados.head()

In [ ]:
conexao = sqlite3.connect(DB_PATH)

dados = pd.read_sql_query("""
    SELECT
        r.id,
        r.colaborador_id,
        r.uid,
        r.evento,
        r.resultado,
        r.registrado_em,
        c.nome,
        c.matricula,
        c.autorizado
    FROM registros_acesso r
    LEFT JOIN colaboradores c
        ON r.colaborador_id = c.id
""", conexao)

dados["registrado_em"] = pd.to_datetime(dados["registrado_em"])
dados["data"] = dados["registrado_em"].dt.strftime("%Y-%m-%d")

dados.head()

In [ ]:
dia = "2026-10-28"

dados_dia = dados[dados["data"] == dia]

print("Data analisada:")
print(dia)

print("Total de registros no dia:")
print(dados_dia.shape[0])

# Selecionando os registros do dia que será analisado

In [ ]:
# Quantas pessoas entraram na sala em determinado dia?

print("Pessoas que entraram na sala:")
print(
    dados_dia[dados_dia["evento"] == "entrada"]
    ["colaborador_id"].nunique()
)

# Contando quantos colaboradores diferentes entraram na sala no dia

In [ ]:
# Quantas pessoas saíram da sala em determinado dia?

print("Pessoas que saíram da sala:")
print(
    dados_dia[dados_dia["evento"] == "saida"]
    ["colaborador_id"].nunique()
)

# Contando quantos colaboradores diferentes saíram da sala no dia

In [ ]:
# Quanto tempo um colaborador específico permaneceu na sala em um determinado dia?

matricula = "MAT001"

colaborador = dados_dia[dados_dia["matricula"] == matricula]

entradas = colaborador[
    colaborador["evento"] == "entrada"
]["registrado_em"].tolist()

saidas = colaborador[
    colaborador["evento"] == "saida"
]["registrado_em"].tolist()

tempo_total = pd.Timedelta(0)

for entrada, saida in zip(entradas, saidas):
    if saida >= entrada:
        tempo_total += saida - entrada

print("Matrícula:", matricula)
print("Dia:", dia)
print("Tempo total na sala:", tempo_total)

# Calculando o tempo de permanência do colaborador no dia

In [ ]:
# Quantas tentativas de acesso negado ocorreram em um determinado dia?

print("Tentativas de acesso negado:")
print(
    dados_dia[dados_dia["resultado"] == "negado"].shape[0]
)

# Contando as tentativas de acesso negado no dia

In [ ]:
# Quantas tentativas de invasão ocorreram em um determinado dia?

print("Tentativas com tags desconhecidas:")
print(
    dados_dia[dados_dia["evento"] == "desconhecido"].shape[0]
)

# Contando os eventos de tags desconhecidas no dia

In [ ]:
# Quais colaboradores não autorizados mais tentaram acessar a sala em um determinado dia?

nao_autorizados = dados_dia[
    (dados_dia["evento"] == "negado") &
    (dados_dia["autorizado"] == 0)
]

print("Tentativas de acesso por colaboradores não autorizados:")
print(nao_autorizados["nome"].value_counts())

# Identificando quais colaboradores não autorizados mais tentaram acessar a sala

In [ ]:
conexao.close()

# Encerrando a conexão com o banco de dados